In [2]:
import pandas as pd

df = pd.read_excel(
    "../data/ASSISTments_mastery_prediction_training.csv.xlsx"
)

print(df.shape)
print(df.columns.tolist())

(270183, 23)
['source_dataset', 'learner_id', 'interaction_index', 'skill_id', 'topic_name', 'correct', 'attempt_count', 'answer_type', 'prior_interactions', 'prior_correct_count', 'prior_accuracy', 'recent_accuracy_5', 'prior_mean_attempt_count', 'prior_skill_attempts', 'prior_skill_correct_count', 'prior_skill_accuracy', 'next_interaction_correct', 'mastery_proxy_after_interaction', 'recommended_action_rule', 'response_time_seconds', 'hint_used', 'has_response_time', 'has_hint_usage']


In [3]:
print(df["next_interaction_correct"].value_counts())
print()
print(df["next_interaction_correct"].value_counts(normalize=True) * 100)

next_interaction_correct
1    179209
0     90974
Name: count, dtype: int64

next_interaction_correct
1    66.328748
0    33.671252
Name: proportion, dtype: float64


In [4]:
features = [
    "prior_interactions",
    "prior_accuracy",
    "recent_accuracy_5",
    "prior_mean_attempt_count",
    "prior_skill_attempts",
    "prior_skill_accuracy"
]

print(df[features].info())

<class 'pandas.DataFrame'>
RangeIndex: 270183 entries, 0 to 270182
Data columns (total 6 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   prior_interactions        270183 non-null  int64  
 1   prior_accuracy            266177 non-null  float64
 2   recent_accuracy_5         266177 non-null  float64
 3   prior_mean_attempt_count  266177 non-null  float64
 4   prior_skill_attempts      270183 non-null  int64  
 5   prior_skill_accuracy      230182 non-null  float64
dtypes: float64(4), int64(2)
memory usage: 12.4 MB
None


In [5]:
print(df[features].describe())

       prior_interactions  prior_accuracy  recent_accuracy_5  \
count       270183.000000   266177.000000      266177.000000   
mean           158.335858        0.667099           0.659962   
std            178.848539        0.201548           0.292619   
min              0.000000        0.000000           0.000000   
25%             25.000000        0.574324           0.400000   
50%             85.000000        0.708333           0.800000   
75%            239.000000        0.800000           0.800000   
max           1038.000000        1.000000           1.000000   

       prior_mean_attempt_count  prior_skill_attempts  prior_skill_accuracy  
count             266177.000000         270183.000000         230182.000000  
mean                   1.571830             10.310982              0.579150  
std                    3.001470             16.957864              0.323447  
min                    0.000000              0.000000              0.000000  
25%                    1.175896  

In [6]:
from sklearn.model_selection import train_test_split

# Get the unique learners
learners = df["learner_id"].unique()

# Split learners into training and testing groups
train_learners, test_learners = train_test_split(
    learners,
    test_size=0.20,
    random_state=42
)

# Create the training and testing datasets
train_df = df[df["learner_id"].isin(train_learners)].copy()
test_df = df[df["learner_id"].isin(test_learners)].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("Training learners:", train_df["learner_id"].nunique())
print("Testing learners:", test_df["learner_id"].nunique())

Training rows: 213570
Testing rows: 56613
Training learners: 3204
Testing learners: 802


In [7]:
overlap = set(train_learners) & set(test_learners)

print("Number of learners in both training and testing:", len(overlap))

Number of learners in both training and testing: 0


In [8]:
# Define the predictor variables
features = [
    "prior_interactions",
    "prior_accuracy",
    "recent_accuracy_5",
    "prior_mean_attempt_count",
    "prior_skill_attempts",
    "prior_skill_accuracy"
]

# Define training features and target
X_train = train_df[features]
y_train = train_df["next_interaction_correct"]

# Define testing features and target
X_test = test_df[features]
y_test = test_df["next_interaction_correct"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (213570, 6)
y_train shape: (213570,)
X_test shape: (56613, 6)
y_test shape: (56613,)


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Build the preprocessing and modelling pipeline
logistic_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

print(logistic_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])


In [10]:
# Train the baseline Logistic Regression model
logistic_pipeline.fit(X_train, y_train)

print("Baseline Logistic Regression model trained successfully.")

Baseline Logistic Regression model trained successfully.


In [11]:
# Generate predictions on the test set
y_pred = logistic_pipeline.predict(X_test)

# Generate predicted probabilities for class 1
y_pred_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")
print("Number of predictions:", len(y_pred))

Predictions generated successfully.
Number of predictions: 56613


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

Accuracy:  0.6996
Precision: 0.7072
Recall:    0.9193
F1 Score:  0.7994
ROC-AUC:   0.6935


In [14]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 5724 14030]
 [ 2975 33884]]


In [15]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold Stratified Group Cross-Validation created.")

5-fold Stratified Group Cross-Validation created.


In [16]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    groups=train_df["learner_id"],
    scoring="roc_auc",
    n_jobs=-1
)

print("ROC-AUC scores for each fold:")
print(cv_scores)

print()
print(f"Mean CV ROC-AUC: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")

ROC-AUC scores for each fold:
[0.67501573 0.67639662 0.66794934 0.67218161 0.67509365]

Mean CV ROC-AUC: 0.6733
Standard deviation: 0.0030


In [17]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

print("GridSearchCV configured successfully.")

GridSearchCV configured successfully.


In [19]:
# Run hyperparameter tuning
grid_search.fit(
    X_train,
    y_train,
    groups=train_df["learner_id"]
)

print("Hyperparameter tuning completed.")

Hyperparameter tuning completed.


In [20]:
print("Best parameters:")
print(grid_search.best_params_)

print()
print(f"Best mean CV ROC-AUC: {grid_search.best_score_:.4f}")

Best parameters:
{'model__C': 100, 'model__class_weight': 'balanced'}

Best mean CV ROC-AUC: 0.6737


In [21]:
# Get the best tuned model
best_model = grid_search.best_estimator_

# Generate predictions on the test set
y_pred_tuned = best_model.predict(X_test)

# Generate predicted probabilities for class 1
y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]

print("Tuned model predictions generated successfully.")
print("Number of predictions:", len(y_pred_tuned))

Tuned model predictions generated successfully.
Number of predictions: 56613


In [22]:
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned)
recall_tuned = recall_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned)
roc_auc_tuned = roc_auc_score(y_test, y_pred_proba_tuned)

print(f"Accuracy:  {accuracy_tuned:.4f}")
print(f"Precision: {precision_tuned:.4f}")
print(f"Recall:    {recall_tuned:.4f}")
print(f"F1 Score:  {f1_tuned:.4f}")
print(f"ROC-AUC:   {roc_auc_tuned:.4f}")

Accuracy:  0.6556
Precision: 0.7630
Recall:    0.6833
F1 Score:  0.7209
ROC-AUC:   0.6938


In [23]:
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

print("Tuned Model Confusion Matrix:")
print(cm_tuned)

Tuned Model Confusion Matrix:
[[11930  7824]
 [11675 25184]]


In [24]:
import pandas as pd

# Save model performance results
results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Baseline": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ],
    "Tuned": [
        accuracy_tuned,
        precision_tuned,
        recall_tuned,
        f1_tuned,
        roc_auc_tuned
    ]
})

results.to_csv("../outputs/model_results.csv", index=False)

# Save cross-validation results
cv_results_summary = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "ROC-AUC": cv_scores
})

cv_results_summary.to_csv(
    "../outputs/baseline_cv_results.csv",
    index=False
)

# Save GridSearch results
grid_results = pd.DataFrame(grid_search.cv_results_)
grid_results.to_csv(
    "../outputs/grid_search_results.csv",
    index=False
)

print("Results saved successfully.")

Results saved successfully.


In [25]:
predictions = pd.DataFrame({
    "actual": y_test,
    "predicted_baseline": y_pred,
    "predicted_tuned": y_pred_tuned,
    "probability_baseline": y_pred_proba,
    "probability_tuned": y_pred_proba_tuned
})

predictions.to_csv(
    "../outputs/test_predictions.csv",
    index=False
)

print("Test predictions saved successfully.")

Test predictions saved successfully.
